# 01 – Nettoyage et vérification (Sujet 9 : Groupe Valmont)

Objectif : passer de 940 lignes brutes à 800 lignes sans doublons, puis à 794 lignes pour la modélisation, en documentant chaque décision.

In [ ]:
import pandas as pd
import numpy as np

hotels = pd.read_csv('../data/raw/Valmont_hotels.csv')
brut = pd.read_csv('../data/raw/Valmont_reservations.csv')
print(hotels.shape, brut.shape)
brut.head()

## 1. Inspection
On regarde les types, les manquants et les valeurs extrêmes avant de toucher à quoi que ce soit.

In [ ]:
print(brut.dtypes)
print(brut.isna().sum()[lambda s: s > 0])
brut.describe().T

## 2. Doublons (940 → 800)
On compte d'abord les doublons exacts, puis on vérifie que les identifiants répétés correspondent bien à des lignes strictement identiques (sinon ce serait un autre problème qu'un simple doublon).

In [ ]:
n_exacts = brut.duplicated().sum()
n_id = brut['reservation_id'].duplicated().sum()
print('doublons exacts :', n_exacts, '| identifiants répétés :', n_id)
# si les deux nombres sont égaux, chaque id répété est un doublon exact
assert n_exacts == n_id
df = brut.drop_duplicates().reset_index(drop=True)
print('lignes après dédoublonnage :', len(df))

## 3. Cohérence avec la table des hôtels
Chaque réservation doit pointer vers un hôtel connu (clé de jointure pour Q3, Q4, Q6, Q9).

In [ ]:
print('hotel_id inconnus :', set(df.hotel_id) - set(hotels.hotel_id))
print('hôtels sans réservation :', len(set(hotels.hotel_id) - set(df.hotel_id)))
print(hotels.categorie.value_counts())

## 4. Valeurs manquantes
Seules `nb_personnes` (20) et `petit_dejeuner` (20) sont concernées.

**Décisions :**
- `nb_personnes` : médiane **par segment déclaré**, car la taille d'un séjour dépend très fortement du segment (groupe ≈ 11, loisir ≈ 2, affaires ≈ 1). Une médiane globale (1) sous-estimerait les groupes.
- `petit_dejeuner` : modalité `inconnu` plutôt qu'une valeur inventée (variable binaire sans règle d'imputation fiable).
- On ajoute un indicateur `nb_personnes_impute` pour pouvoir tester la sensibilité plus tard.

In [ ]:
print(df.groupby('segment_declare')['nb_personnes'].median())
df['nb_personnes_impute'] = df['nb_personnes'].isna()
df['nb_personnes'] = df['nb_personnes'].fillna(
    df.groupby('segment_declare')['nb_personnes'].transform('median'))
df['petit_dejeuner'] = df['petit_dejeuner'].fillna('inconnu')
print(df[['nb_personnes', 'petit_dejeuner']].isna().sum())

## 5. Valeurs aberrantes à signaler : 365 nuits et prix à 0 €
On **ne supprime pas en silence** : on les isole dans un fichier à part (réutilisé en Q9) et on les exclut seulement de la base de modélisation.

In [ ]:
masque = (df.nb_nuits == 365) | (df.prix_nuit_eur == 0)
print('séjours de 365 nuits :', (df.nb_nuits == 365).sum())
print('prix à 0 EUR :', (df.prix_nuit_eur == 0).sum())
print('lignes concernées (union) :', masque.sum())
anomalies = df[masque].copy()
anomalies['motif'] = np.where(anomalies.nb_nuits == 365, '365 nuits', 'prix 0 EUR')
anomalies

## 6. Colonne `nuitees_cumulees_client`
Un cumul client devrait être **supérieur ou égal** au nombre de nuits de la réservation en cours. On mesure les cas incohérents.

**Décision :** on garde la colonne dans le fichier mais on ne l'utilise pas sans vérification. Pour Q5, on compare le modèle avec et sans (risque de fuite d'information, la valeur pouvant être calculée après coup).

In [ ]:
df['cumul_incoherent'] = df['nuitees_cumulees_client'] < df['nb_nuits']
print('lignes incohérentes :', df['cumul_incoherent'].sum())
print(df.loc[df.cumul_incoherent, ['nb_nuits', 'nuitees_cumulees_client', 'a_annule']].head())
print(df.groupby('cumul_incoherent')['a_annule'].apply(lambda s: (s == 'oui').mean()))

## 7. Fichiers de sortie et récapitulatif

In [ ]:
modelisation = df[~masque].copy()
df.to_csv('../data/clean/reservations_propres_800.csv', index=False)
modelisation.to_csv('../data/clean/reservations_modelisation_794.csv', index=False)
anomalies.to_csv('../data/clean/anomalies_a_part.csv', index=False)

recap = pd.DataFrame({'étape': ['Lignes brutes', 'Après suppression des doublons', 'Base de modélisation'],
                      'lignes': [len(brut), len(df), len(modelisation)]})
print(recap)
print('Taux global d\'annulation (800) :', round((df.a_annule == 'oui').mean(), 3))
